# A/B-тест: анализ ecommerce-эксперимента

## Бизнес-контекст
Интернет-магазин протестировал новый дизайн страницы.

**Бизнес-вопрос:** стоит ли раскатывать новый дизайн на всех пользователей?

## Primary metric
**Conversion Rate (CR)** — доля пользователей, совершивших целевое действие.

## Гипотезы
- **H0:** новый дизайн не увеличивает конверсию относительно контрольной группы.
- **H1:** новый дизайн увеличивает конверсию относительно контрольной группы.

Уровень значимости: **alpha = 0.05**.

Направление гипотезы задаётся до анализа результатов: нас интересует именно рост конверсии, а не любое отличие.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')

In [3]:
df = pd.read_csv('ab_data.csv')

print('Размер датасета:', df.shape)
print('\nТипы данных:')
print(df.dtypes)
print('\nПервые строки:')
df.head()

Размер датасета: (294480, 5)

Типы данных:
user_id          int64
timestamp       object
group           object
landing_page    object
converted        int64
dtype: object

Первые строки:


,user_id,timestamp,group,landing_page,converted
0,851104,11:48.6,control,old_page,0
1,804228,01:45.2,control,old_page,0
2,661590,55:06.2,treatment,new_page,0
3,853541,28:03.1,treatment,new_page,0
4,864975,52:26.2,control,old_page,1


In [4]:
print('Пропуски:')
print(df.isnull().sum())

print('\nУникальные значения в группах:')
print(df['group'].value_counts())

print('\nУникальные значения в landing_page:')
print(df['landing_page'].value_counts())

Пропуски:
user_id         0
timestamp       0
group           0
landing_page    0
converted       0
dtype: int64

Уникальные значения в группах:
group
treatment    147278
control      147202
Name: count, dtype: int64

Уникальные значения в landing_page:
landing_page
new_page    147241
old_page    147239
Name: count, dtype: int64


In [5]:
# Несоответствия: treatment видит old_page или control видит new_page
mismatch = df[((df['group'] == 'treatment') & (df['landing_page'] == 'old_page')) |
              ((df['group'] == 'control') & (df['landing_page'] == 'new_page'))]

print('Количество несоответствий:', len(mismatch))
print('Это', round(len(mismatch) / len(df) * 100, 2), '% от всех данных')

Количество несоответствий: 3893
Это 1.32 % от всех данных


## Проблема в данных: несоответствие группы и страницы

Обнаружено 3893 строки (1.32%), где пользователь из группы `treatment` 
видел `old_page` или из `control` видел `new_page`.

Решение: удалить, а не исправлять — причина несоответствия неизвестна, 
любая правка была бы домыслом и исказила бы результаты теста.

In [6]:
# Удаляем строки с несоответствием группы и страницы
df_clean = df[((df['group'] == 'treatment') & (df['landing_page'] == 'new_page')) |
              ((df['group'] == 'control') & (df['landing_page'] == 'old_page'))]

print('Было строк:', len(df))
print('Стало строк:', len(df_clean))
print('Удалено:', len(df) - len(df_clean))

Было строк: 294480
Стало строк: 290587
Удалено: 3893


In [7]:
# Проверяем дубликаты по user_id
print('Всего пользователей:', df_clean['user_id'].nunique())
print('Всего строк:', len(df_clean))
print('Дубликаты:', len(df_clean) - df_clean['user_id'].nunique())

Всего пользователей: 290585
Всего строк: 290587
Дубликаты: 2


In [18]:
df_clean = df_clean.drop_duplicates(subset=['user_id'])
print('Строк после удаления дубликатов:', len(df_clean))

Строк после удаления дубликатов: 290585


## Итог чистки данных

Исходный датасет после удаления несоответствий: 290 587 строк.
Обнаружено 2 дубликата по `user_id` — один пользователь попал в обе группы,
что искажает результаты теста. Дубликаты удалены.

Финальный датасет: 290 585 строк.

In [20]:
converted_mean = df_clean.groupby("group")["converted"].mean()
print(converted_mean)

group
control      0.120386
treatment    0.118807
Name: converted, dtype: float64


In [25]:
import seaborn as sns
import matplotlib.pyplot as plt

conversion_by_group = df_clean.groupby('group')['converted'].mean().reset_index()
plt.figure(figsize=(10, 6))
sns.barplot(data=conversion_by_group, x='group', y='converted')
plt.title('Conversion Rate by Group', fontsize=14)
plt.xlabel('Group', fontsize=12)
plt.ylabel('Conversion Rate', fontsize=12)
plt.ylim(0.10, 0.13)


(0.1, 0.13)

## Конверсия по группам

- Контроль (старый дизайн): 12.04%
- Тест (новый дизайн): 11.88%

Визуально конверсия в контроле выше, но разница минимальная — 0.16%.
Необходимо проверить статистическую значимость этой разницы.

In [42]:
from statsmodels.stats.proportion import proportions_ztest



count = df_clean.groupby("group")["converted"].sum()
nobs = df_clean.groupby("group")["converted"].count()
z_stat, p_value = proportions_ztest(count, nobs)

print(f"Z-статистика: {z_stat:.4f}")
print(f"p-значение: {p_value:.4f}")

Z-статистика: 1.3116
p-значение: 0.1897


## Результаты z-test

- **Z-statistic:** 1.31
- **P-value:** 0.19
- **alpha:** 0.05

Так как **p-value > alpha**, статистически значимого различия между группами не обнаружено.

Мы **не отвергаем H0**. Это не означает, что H0 доказана; данных недостаточно, чтобы утверждать, что новый дизайн увеличивает конверсию.

Важно: p-value не является вероятностью того, что H0 истинна.

## 95% Confidence Interval

95% доверительный интервал показывает диапазон правдоподобных значений истинной разницы в конверсии.

Если интервал содержит 0, статистически значимого различия не обнаружено.

In [ ]:
from statsmodels.stats.proportion import confint_proportions_2indep

# Считаем разницу конверсий как Treatment - Control
count = df_clean.groupby("group")["converted"].sum()
nobs = df_clean.groupby("group")["converted"].count()

ci_low, ci_high = confint_proportions_2indep(
    count1=count["treatment"],
    nobs1=nobs["treatment"],
    count2=count["control"],
    nobs2=nobs["control"],
    method="newcomb",
    compare="diff",
    alpha=0.05
)

effect = (count["treatment"] / nobs["treatment"]) - (count["control"] / nobs["control"])

print(f"Observed effect (Treatment - Control): {effect * 100:.2f} п.п.")
print(f"95% CI: [{ci_low * 100:.2f}; {ci_high * 100:.2f}] п.п.")


## MDE и чувствительность эксперимента

MDE (Minimum Detectable Effect) — минимальный эффект, который эксперимент способен обнаружить при заданных alpha, power и размере выборки.

Параметры:
- alpha = 0.05
- Power = 80%
- baseline conversion = контрольная группа
- sample size = фактический размер групп

In [ ]:
from scipy.stats import norm

alpha = 0.05
power = 0.80
p_baseline = count["control"] / nobs["control"]
n_eff = 2 / (1 / nobs["control"] + 1 / nobs["treatment"])

z_alpha = norm.ppf(1 - alpha / 2)
z_power = norm.ppf(power)

mde = (z_alpha + z_power) * np.sqrt(
    2 * p_baseline * (1 - p_baseline) / n_eff
)

print(f"MDE: {mde*100:.2f} п.п.")

## Practical significance

Наблюдаемое изменение: **−0.16 п.п.**

MDE показывает минимальный положительный эффект, который тест способен обнаруживать при выбранной мощности. Наблюдаемое изменение не является улучшением конверсии.

## Effect visualization

График показывает наблюдаемую разницу и 95% CI. Нулевая линия означает отсутствие эффекта.

Если CI пересекает 0, данных недостаточно для вывода о статистически значимом изменении.

In [ ]:
import matplotlib.pyplot as plt

effect = (count["treatment"] / nobs["treatment"]) - (count["control"] / nobs["control"])

plt.figure(figsize=(9, 4))
plt.errorbar(
    x=[effect * 100],
    y=[0],
    xerr=[[
        (effect - ci_low) * 100,
        (ci_high - effect) * 100
    ]],
    fmt="o",
    capsize=6
)
plt.axvline(0, linestyle="--")
plt.xlabel("Treatment - Control, percentage points")
plt.yticks([])
plt.title("Conversion effect with 95% confidence interval")
plt.show()

## Guardrails и ограничения данных

В полноценном продуктовом эксперименте одного CR недостаточно. Перед раскаткой обычно заранее задаются guardrail-метрики, например:
- AOV / средний чек;
- ошибки оплаты;
- возвраты;
- время до целевого действия.

**Ограничение текущего датасета:** в нём нет этих бизнес-метрик, поэтому мы не можем честно посчитать их и не будем придумывать значения.

Следовательно, финальное решение в этом проекте основано на **конверсии, статистической неопределённости, размере эффекта и чувствительности теста**.

## Проверка качества эксперимента: баланс групп (SRM)

Перед интерпретацией результата проверим, соответствует ли распределение пользователей ожидаемому **50/50**.

Если группы сильно отличаются от ожидаемого соотношения, это может указывать на проблему с рандомизацией или сбором данных. Это называется **Sample Ratio Mismatch (SRM)**.

In [ ]:
from scipy.stats import chisquare

group_counts = df_clean['group'].value_counts()
expected = [len(df_clean) / 2, len(df_clean) / 2]

srm_stat, srm_p = chisquare(
    f_obs=[group_counts['control'], group_counts['treatment']],
    f_exp=expected
)

print('Control:', group_counts['control'])
print('Treatment:', group_counts['treatment'])
print(f'SRM p-value: {srm_p:.4f}')

### Как читать результат

Если **p-value < 0.05**, распределение групп отличается от ожидаемого 50/50 статистически значимо — тогда причину SRM нужно расследовать до интерпретации A/B-теста.

Если **p-value ≥ 0.05**, статистически значимого нарушения ожидаемого соотношения не обнаружено. Это не доказывает идеальную рандомизацию, но снимает один из важных рисков качества эксперимента.

## Время проведения эксперимента

Проверим период эксперимента и посмотрим, насколько стабильно ведут себя группы по дням. Это важно, потому что общий результат может скрывать временные изменения поведения пользователей.

In [ ]:
df_clean['timestamp_dt'] = pd.to_datetime(df_clean['timestamp'])

experiment_start = df_clean['timestamp_dt'].min()
experiment_end = df_clean['timestamp_dt'].max()
duration_days = (experiment_end - experiment_start).total_seconds() / 86400

print('Начало:', experiment_start)
print('Конец:', experiment_end)
print(f'Продолжительность: {duration_days:.2f} дней')

In [ ]:
daily_conversion = (
    df_clean.assign(date=df_clean['timestamp_dt'].dt.date)
    .groupby(['date', 'group'])['converted']
    .mean()
    .reset_index()
)

plt.figure(figsize=(10, 5))
for group in ['control', 'treatment']:
    part = daily_conversion[daily_conversion['group'] == group]
    plt.plot(part['date'], part['converted'] * 100, marker='o', label=group)

plt.title('Daily Conversion Rate by Experiment Group')
plt.xlabel('Date')
plt.ylabel('Conversion Rate, %')
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

### Зачем нам этот график

Мы не пытаемся доказать причинность по дневному графику. Наша задача — найти явные проблемы: резкие провалы, пропуски или периоды, где поведение групп резко меняется.

Если такие участки есть, их нужно исследовать отдельно, прежде чем делать бизнес-вывод.

### Результаты проверки качества

- **SRM p-value = 0.9453** → статистически значимого нарушения ожидаемого соотношения 50/50 не обнаружено.
- Эксперимент длился примерно **22 дня**: с 2 января по 24 января 2017 года.
- Дневной график используем как диагностическую проверку: он помогает заметить аномалии во времени, но сам по себе не доказывает эффект нового дизайна.

## Динамика по дням: дополнительная проверка

Помимо самой конверсии, проверим размер выборки по дням. Если в отдельный день пользователей слишком мало, дневная конверсия может сильно колебаться случайно.

Поэтому дневной график используем как **диагностический инструмент**, а не как отдельный A/B-тест.

In [ ]:
daily_stats = (
    df_clean.assign(date=df_clean['timestamp_dt'].dt.date)
    .groupby(['date', 'group'])
    .agg(
        users=('user_id', 'nunique'),
        conversions=('converted', 'sum'),
        conversion_rate=('converted', 'mean')
    )
    .reset_index()
)

daily_stats['conversion_rate'] = daily_stats['conversion_rate'] * 100

daily_stats.head()

In [ ]:
print('Минимальный размер дневной выборки:')
print(daily_stats.groupby('group')['users'].min())

print('\nМаксимальная дневная конверсия:')
print(daily_stats.groupby('group')['conversion_rate'].max().round(2))

print('\nМинимальная дневная конверсия:')
print(daily_stats.groupby('group')['conversion_rate'].min().round(2))

### Интерпретация

Сильные дневные колебания сами по себе не означают, что эффект нового дизайна меняется. При меньшем числе наблюдений дневная конверсия естественно становится более шумной.

Поэтому основной вывод остаётся основанным на **агрегированном эффекте, доверительном интервале и статистическом тесте**, а дневная динамика используется для поиска потенциальных аномалий.

## Проверка стабильности эффекта во времени

Общий результат эксперимента показывает разницу между группами. Теперь проверим, сохраняется ли направление эффекта в первой и второй половине эксперимента.

Это **не отдельный подтверждающий A/B-тест**, а диагностический анализ: маленькие подгруппы имеют больше неопределённости.

In [ ]:
df_clean['period'] = pd.qcut(
    df_clean['timestamp_dt'],
    q=2,
    labels=['first_half', 'second_half']
)

period_summary = (
    df_clean.groupby(['period', 'group'], observed=True)['converted']
    .agg(['mean', 'count', 'sum'])
    .reset_index()
)

period_summary['conversion_rate'] = period_summary['mean'] * 100
period_summary[['period', 'group', 'conversion_rate', 'count', 'sum']]

In [ ]:
period_pivot = period_summary.pivot(
    index='period', columns='group', values='conversion_rate'
)
period_pivot['effect_pp'] = period_pivot['treatment'] - period_pivot['control']

period_pivot.round(3)

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(period_pivot.index.astype(str), period_pivot['effect_pp'])
plt.axhline(0, linewidth=1)
plt.ylabel('Effect: Treatment - Control, п.п.')
plt.xlabel('Период эксперимента')
plt.title('Разница конверсии между группами по периодам')
plt.tight_layout()
plt.show()

### Как читать этот график

- значение выше 0 означает, что в этом периоде Treatment имел более высокую конверсию;
- значение ниже 0 означает, что Treatment имел более низкую конверсию;
- если знак эффекта меняется, общий результат может зависеть от временной структуры данных;
- этот анализ помогает найти нестабильность, но сам по себе не доказывает наличие или отсутствие эффекта.

Основной вывод по-прежнему делаем по всей выборке эксперимента.

## Decision summary

| Критерий | Результат | Интерпретация |
|---|---:|---|
| Control CR | 12.04% | Базовый уровень |
| Treatment CR | 11.88% | Ниже контроля |
| Absolute effect | −0.16 п.п. | Наблюдаемое снижение |
| Relative change | −1.31% | Небольшое относительное снижение |
| p-value | 0.19 | Нет статистически значимого различия |
| 95% CI | примерно [−0.40; +0.08] п.п. | Включает 0 |
| Cohen's h | 0.0049 | Очень малый стандартизированный эффект |
| MDE | примерно +0.34 п.п. | Ориентир чувствительности теста |

### Decision
**Не раскатывать новый дизайн на 100% пользователей на основании текущего эксперимента.**

Причина не в том, что новый дизайн доказанно хуже, а в том, что эксперимент не подтвердил необходимое улучшение конверсии.

In [43]:

from statsmodels.stats.proportion import proportion_effectsize

p1 = converted_mean["control"]
p2 = converted_mean["treatment"]

h = proportion_effectsize(p1, p2)
print(f"Cohen's h = {h:.4f}")

Cohen's h = 0.0049


## Effect Size — Cohen's h

Cohen's h = 0.0049 — эффект практически нулевой.

Даже если бы разница оказалась статистически значимой,
на практике она настолько мала, что не имеет смысла для бизнеса.

## Итоговый вывод

### Data quality
- 3893 строки с несоответствием группы и страницы — удалены.
- 2 дубликата по user_id — удалены.
- Финальный датасет: **290 585 пользователей**.

### Experiment results
- Control: **12.04%**
- Treatment: **11.88%**
- Absolute difference: **−0.16 п.п.**
- Relative change: **−1.31%**
- Two-sided p-value: **0.19**
- Cohen's h: **0.0049**
- 95% CI: рассчитывается ниже и включает 0.
- MDE: рассчитывается ниже при alpha=0.05 и power=80%.

### Business decision

Статистически значимого увеличения конверсии нового дизайна не обнаружено.

**Не раскатывать новый дизайн на 100% пользователей по результатам текущего эксперимента.**

Это не означает, что новый дизайн доказанно хуже. Это означает, что текущий эксперимент не подтвердил необходимое улучшение конверсии.